# Hallucination Probe - Stages 0 to 5

Detecting when a language model is about to be wrong, by reading its own internal activations.

**Runtime -> Change runtime type -> T4 GPU** before you start.

Run the cells in order. You will know whether the idea works by the end of Stage 3.

## Stage 0 - Setup

In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets scikit-learn matplotlib

import json, re, string, numpy as np, torch
from pathlib import Path
from tqdm.auto import tqdm

WORK = Path("/content/drive/MyDrive/hallucination_probe")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    WORK = Path("/content/hallucination_probe")
    print("no Drive - using local storage (lost on disconnect)")
WORK.mkdir(parents=True, exist_ok=True)
print("working dir:", WORK)

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # ungated: no access request needed
N_EXAMPLES = 2000
MAX_NEW_TOKENS = 20
SHARD_SIZE = 250

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"          # required for correct batched generation

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16))
model.eval()
print(model.config.num_hidden_layers, "layers, hidden dim", model.config.hidden_size)

## Stage 1 - Ask 2000 questions and grade the answers

We need labels: *the model said X, and X was wrong.*

In [ ]:
from datasets import load_dataset
ds = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation")
ds = ds.select(range(N_EXAMPLES))
print(ds[0]["question"], "->", ds[0]["answer"]["aliases"][:3])

In [ ]:
PUNCT = set(string.punctuation)

def normalize(s):
    s = s.lower()
    s = "".join(c for c in s if c not in PUNCT)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def is_correct(pred, aliases):
    p = normalize(pred)
    if not p:
        return False
    return any(len(normalize(a)) >= 3 and normalize(a) in p for a in aliases)

def build_prompt(question):
    msgs = [{"role": "user",
             "content": "Answer with just the answer, no explanation.\nQ: " + question + "\nA:"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

In [ ]:
ANSWERS = WORK / "answers.jsonl"
done = sum(1 for _ in open(ANSWERS)) if ANSWERS.exists() else 0
print("resuming from", done)

BATCH = 8
out = open(ANSWERS, "a", encoding="utf-8")
for start in tqdm(range(done, len(ds), BATCH)):
    rows = [ds[i] for i in range(start, min(start + BATCH, len(ds)))]
    prompts = [build_prompt(r["question"]) for r in rows]
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=False, pad_token_id=tok.pad_token_id)
    answers = tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    for r, p, a in zip(rows, prompts, answers):
        aliases = r["answer"]["aliases"] or [r["answer"]["value"]]
        a = a.strip()
        out.write(json.dumps({"question": r["question"], "prompt": p, "answer": a,
                              "aliases": aliases,
                              "label": 0 if is_correct(a, aliases) else 1,
                              "group": normalize(aliases[0])}, ensure_ascii=False) + "\n")
    out.flush()
out.close()

recs = [json.loads(l) for l in open(ANSWERS, encoding="utf-8")]
rate = float(np.mean([r["label"] for r in recs]))
print(len(recs), "graded - hallucination rate", round(rate, 3),
      "(accuracy", round(1 - rate, 3), ")")
print("SANITY: expect 40-60% accuracy. Above 95% or below 5% means grading is broken.")

### Audit the grading before trusting anything downstream

In [ ]:
import random
random.seed(0)
for i, r in enumerate(random.sample(recs, 15), 1):
    print("[", i, "] graded", "WRONG" if r["label"] else "right")
    print("    Q:", r["question"])
    print("    model:", repr(r["answer"]))
    print("    gold :", r["aliases"][:3], "\n")
print("Count your disagreements. That is your label-noise rate - report it.")

## Stage 2 - Capture the internal activations

Two passes: generation already happened, so now we re-run one forward pass over
prompt+answer and read the hidden states. Much simpler than extracting
mid-generation.

**This is the step that makes the project cheap.** Once these files exist, every
later experiment runs on CPU in seconds.

In [ ]:
@torch.no_grad()
def activations_for(prompt, answer):
    n_prompt = tok(prompt, return_tensors="pt").input_ids.shape[1]
    enc = tok(prompt + answer, return_tensors="pt").to(model.device)
    n_total = enc.input_ids.shape[1]
    if n_total <= n_prompt:
        return None
    hs = model(**enc, output_hidden_states=True).hidden_states
    # mean-pool over the ANSWER tokens only, one vector per layer
    return np.stack([h[0, n_prompt:n_total, :].mean(0).float().cpu().numpy() for h in hs])

In [ ]:
labels, groups = [], []
for s in range(0, len(recs), SHARD_SIZE):
    sid = s // SHARD_SIZE
    path = WORK / ("acts_%05d.npy" % sid)
    chunk = recs[s:s + SHARD_SIZE]
    if path.exists():
        print("shard", sid, "exists - skipping")
        labels += [r["label"] for r in chunk]
        groups += [r["group"] for r in chunk]
        continue
    feats, kl, kg = [], [], []
    for r in tqdm(chunk, desc="shard %d" % sid):
        a = activations_for(r["prompt"], r["answer"])
        if a is None:
            continue
        feats.append(a); kl.append(r["label"]); kg.append(r["group"])
    np.save(path, np.stack(feats).astype(np.float16))
    labels += kl; groups += kg

np.save(WORK / "labels.npy", np.array(labels))
np.save(WORK / "groups.npy", np.array(groups, dtype=object), allow_pickle=True)
print("done - GPU no longer needed from here on")

## Stage 3 - Train the probe

The moment of truth. Note the **entity-grouped split**: a random split would put
the same entity in train and test, and the probe would learn entity frequency
instead of truthfulness.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X = np.concatenate([np.load(p) for p in sorted(WORK.glob("acts_*.npy"))])
y = np.load(WORK / "labels.npy")
g = np.load(WORK / "groups.npy", allow_pickle=True)
n = min(len(X), len(y), len(g)); X, y, g = X[:n], y[:n], g[:n]
print("X", X.shape, " hallucination rate", round(float(y.mean()), 3))

tr, te = next(GroupShuffleSplit(1, test_size=0.3, random_state=0)
              .split(np.zeros(len(y)), y, g))
print("train", len(tr), "/ test", len(te), "(entity-disjoint)")

def fit(Xtr, ytr):
    return make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=3000)).fit(Xtr, ytr)

L = X.shape[1] // 2                      # start mid-network
Xl = X[:, L, :].astype(np.float32)
auc = roc_auc_score(y[te], fit(Xl[tr], y[tr]).predict_proba(Xl[te])[:, 1])
print("\nlayer", L, " AUROC", round(auc, 4))
print("0.50 chance | 0.65-0.75 real | 0.75-0.85 strong | above 0.95 suspect a leak")

## Stage 4 - Which layer knows best?

In [ ]:
scores = []
for Li in range(X.shape[1]):
    Xi = X[:, Li, :].astype(np.float32)
    scores.append(roc_auc_score(y[te], fit(Xi[tr], y[tr]).predict_proba(Xi[te])[:, 1]))
    print("  layer %3d  %.4f" % (Li, scores[-1]))

best = int(np.argmax(scores))
print("\nBEST layer", best, "- AUROC", round(scores[best], 4),
      "(%.0f%% depth)" % (100 * best / (X.shape[1] - 1)))

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(scores, marker="o", ms=3, color="#1b3a5c")
plt.axhline(0.5, ls="--", lw=1, c="#999", label="chance")
plt.axvline(best, ls=":", lw=1.2, c="#a4551b", label="best = layer %d" % best)
plt.xlabel("layer"); plt.ylabel("AUROC"); plt.legend(); plt.grid(alpha=.25)
plt.title("Where does the truthfulness signal live?")
plt.tight_layout(); plt.savefig(WORK / "layer_sweep.png", dpi=160); plt.show()

### How much would a careless split have inflated this?

In [ ]:
Xb = X[:, best, :].astype(np.float32)
rng = np.random.default_rng(0); perm = rng.permutation(len(y))
cut = int(0.7 * len(y)); rtr, rte = perm[:cut], perm[cut:]
rauc = roc_auc_score(y[rte], fit(Xb[rtr], y[rtr]).predict_proba(Xb[rte])[:, 1])
print("random split  %.4f" % rauc)
print("entity split  %.4f" % scores[best])
print("inflation     %+.4f  <- what a random split would have hidden" % (rauc - scores[best]))

## Stage 5 - The number that sells the project

In [ ]:
clf = fit(Xb[tr], y[tr])
risk = clf.predict_proba(Xb[te])[:, 1]
wrong = y[te]

order = np.argsort(risk)
acc = np.cumsum(1 - wrong[order]) / np.arange(1, len(order) + 1)
cov = np.arange(1, len(order) + 1) / len(order)
base = 1 - wrong.mean()

print("%10s%11s%9s" % ("coverage", "accuracy", "gain"))
for t in (1.0, .9, .8, .7, .6, .5):
    i = max(0, int(t * len(cov)) - 1)
    print("%9.0f%%%10.1f%%%+8.1f" % (100 * cov[i], 100 * acc[i], 100 * (acc[i] - base)))

i80 = max(0, int(.8 * len(cov)) - 1)
print("\n>>> Declining the riskiest 20%%: accuracy %.1f%% -> %.1f%%"
      % (100 * base, 100 * acc[i80]))

plt.figure(figsize=(7, 4))
plt.plot(100 * cov, 100 * acc, lw=2, c="#1e6b4a")
plt.axhline(100 * base, ls="--", lw=1, c="#a4551b",
            label="answer everything (%.1f%%)" % (100 * base))
plt.xlabel("% of questions answered"); plt.ylabel("accuracy on those answered (%)")
plt.title("Abstaining on risky questions raises accuracy")
plt.legend(); plt.grid(alpha=.25); plt.tight_layout()
plt.savefig(WORK / "risk_coverage.png", dpi=160); plt.show()

## Try it live

In [ ]:
def answer_with_confidence(question, threshold=0.6):
    p = build_prompt(question)
    with torch.no_grad():
        enc = tok(p, return_tensors="pt").to(model.device)
        gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=False, pad_token_id=tok.pad_token_id)
    ans = tok.decode(gen[0][enc.input_ids.shape[1]:], skip_special_tokens=True).strip()
    a = activations_for(p, ans)
    r = float(clf.predict_proba(a[best].reshape(1, -1).astype(np.float32))[0, 1])
    flag = "NOT CONFIDENT" if r > threshold else "confident"
    return "[%s risk=%.2f] %s" % (flag, r, ans)

for q in ["What is the capital of France?",
          "Who won the 1994 Nobel Prize in Literature?",
          "What is the population of Zaqatala district?"]:
    print(q)
    print("  ", answer_with_confidence(q), "\n")

---
## What to do next

1. **Audit your labels** - count disagreements from the Stage 1 audit cell and write the number down.
2. **If AUROC is below 0.60** - try `Qwen/Qwen2.5-3B-Instruct`. If it jumps, the signal emerges with scale, and that is itself a finding.
3. **Try other pooling** - last answer token instead of the mean.
4. **Add a baseline** - mean token log-probability, so you can show the probe beats the free option.
5. **Scale up** - raise `N_EXAMPLES`, then add HaluEval or HotpotQA.